In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\AI-Powered-Demand-Inventory-Intelligence\data\online_retail_UTF8.csv")

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 69.3 MB


In [6]:
#Forcasting Dataset

In [7]:
forecast_df = df.copy()

In [8]:
#Cancelled Oder Dataset 

In [9]:
cancelled_df = forecast_df[
    forecast_df["InvoiceNo"].astype(str).str.startswith("C")
]

In [10]:
cancelled_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [11]:
print("Cancelled Orders:", cancelled_df.shape[0])

Cancelled Orders: 9288


In [12]:
#Removing Cancelled Oder From Forcasting Dataset

In [13]:
forecast_df = forecast_df[
    ~forecast_df["InvoiceNo"].astype(str).str.startswith("C")
]

In [14]:
forecast_df.shape

(532621, 8)

In [15]:
#Missing Value

In [16]:
missing = pd.DataFrame({
    "Missing Values": forecast_df.isnull().sum(),
    "Percentage": round(
        (forecast_df.isnull().sum()/len(forecast_df))*100,
        2
    )
})

missing.sort_values("Percentage", ascending=False)

,Missing Values,Percentage
CustomerID,134697,25.29
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


In [17]:
forecast_df["CustomerID"].isnull().sum()

np.int64(134697)

In [18]:
#Remove
forecast_df = forecast_df.dropna(subset=["CustomerID"])

In [19]:
#Check
forecast_df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [20]:
#Duplicate Rows

In [21]:
forecast_df.duplicated().sum()

np.int64(5192)

In [22]:
#Remove
forecast_df.drop_duplicates(inplace=True)

In [23]:
#check
forecast_df.duplicated().sum()

np.int64(0)

In [24]:
#Quantity

In [25]:
(forecast_df["Quantity"] <= 0).sum()

np.int64(0)

In [26]:
forecast_df = forecast_df[
    forecast_df["Quantity"] > 0
]

In [27]:
#Unit Price

In [28]:
(forecast_df["UnitPrice"] <= 0).sum()

np.int64(40)

In [29]:
#Remove negative unit price


In [30]:
forecast_df = forecast_df[
    forecast_df["UnitPrice"] > 0
]

In [32]:
#ckeck
(forecast_df["UnitPrice"] <= 0).sum()

np.int64(0)

In [33]:
#Date Conversion

In [34]:
forecast_df["InvoiceDate"] = pd.to_datetime(
    forecast_df["InvoiceDate"]
)

#Feature Engineering

In [35]:
#Revenue

In [36]:
forecast_df["Revenue"] = (
    forecast_df["Quantity"] *
    forecast_df["UnitPrice"]
)

In [37]:
#YEar
forecast_df["Year"] = forecast_df["InvoiceDate"].dt.year

In [38]:
#Month
forecast_df["Month"] = forecast_df["InvoiceDate"].dt.month

In [39]:
#day
forecast_df["Day"] = forecast_df["InvoiceDate"].dt.day

In [40]:
#Weekday
forecast_df["Weekday"] = forecast_df["InvoiceDate"].dt.day_name()

In [41]:
#Hour
forecast_df["Hour"] = forecast_df["InvoiceDate"].dt.hour

In [42]:
#weekend
forecast_df["IsWeekend"] = forecast_df["Weekday"].isin(
    ["Saturday", "Sunday"]
)

In [43]:
forecast_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,Day,Weekday,Hour,IsWeekend
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,1,Wednesday,8,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,Wednesday,8,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,1,Wednesday,8,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,Wednesday,8,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,1,Wednesday,8,False


In [44]:
forecast_df.info()

<class 'pandas.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 15 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  str           
 1   StockCode    392692 non-null  str           
 2   Description  392692 non-null  str           
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[us]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  str           
 8   Revenue      392692 non-null  float64       
 9   Year         392692 non-null  int32         
 10  Month        392692 non-null  int32         
 11  Day          392692 non-null  int32         
 12  Weekday      392692 non-null  str           
 13  Hour         392692 non-null  int32         
 14  IsWeekend    392692 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(3), int32(4

In [45]:
forecast_df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
Year           0
Month          0
Day            0
Weekday        0
Hour           0
IsWeekend      0
dtype: int64

In [46]:
forecast_df.shape

(392692, 15)

In [47]:
forecast_df["Revenue"].describe()

count    392692.000000
mean         22.631500
std         311.099224
min           0.001000
25%           4.950000
50%          12.450000
75%          19.800000
max      168469.600000
Name: Revenue, dtype: float64

In [49]:
import os

print(os.getcwd())

C:\Users\ASUS


In [50]:
forecast_df.to_csv(
    r"C:\Users\ASUS\OneDrive\Desktop\AI-Powered-Demand-Inventory-Intelligence\data\cleaned_retail.csv",
    index=False
)

In [51]:
cancelled_df.to_csv(
    r"C:\Users\ASUS\OneDrive\Desktop\AI-Powered-Demand-Inventory-Intelligence\data\cancelled_orders.csv",
    index=False
)